## ML-07 — Baseline Action Score and Top-10 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

import duckdb
import numpy as np
import pandas as pd

from pathlib import Path
from google.colab import userdata

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

# Read your Hugging Face token from Colab Secrets.
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add it through the Colab Secrets panel."
    )

# Connect DuckDB to the FlyRank Hugging Face warehouse.
con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FEB_MAR_TABLE = (
    "read_parquet(["
    f"'{REL}/fact_content_daily_performance/month=2026-02/*.parquet', "
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    "])"
)

print("Connected successfully to the FlyRank warehouse.")

Connected successfully to the FlyRank warehouse.


In [2]:
# Build one row per client-content page.
# February provides the input information.
# March provides only the later evaluation outcome.

model_frame = con.execute(
    f"""
    WITH monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            month,

            COUNT(
                DISTINCT CASE
                    WHEN gsc_data_available IS TRUE
                    THEN report_date
                END
            ) AS gsc_observed_days,

            COUNT(
                DISTINCT CASE
                    WHEN ga4_data_available IS TRUE
                    THEN report_date
                END
            ) AS ga4_observed_days,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                COUNT(
                    DISTINCT CASE
                        WHEN gsc_data_available IS TRUE
                        THEN report_date
                    END
                ),
                0
            ) AS avg_daily_impressions,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_clicks, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                COUNT(
                    DISTINCT CASE
                        WHEN gsc_data_available IS TRUE
                        THEN report_date
                    END
                ),
                0
            ) AS avg_daily_clicks,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_sum_position, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                SUM(
                    CASE
                        WHEN gsc_data_available IS TRUE
                        THEN COALESCE(gsc_impressions, 0)
                        ELSE 0
                    END
                ),
                0
            ) AS weighted_avg_position,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN COALESCE(ga4_sessions, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                COUNT(
                    DISTINCT CASE
                        WHEN ga4_data_available IS TRUE
                        THEN report_date
                    END
                ),
                0
            ) AS avg_daily_sessions,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN COALESCE(ga4_engaged_sessions, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                SUM(
                    CASE
                        WHEN ga4_data_available IS TRUE
                        THEN COALESCE(ga4_sessions, 0)
                        ELSE 0
                    END
                ),
                0
            ) AS engagement_rate

        FROM {FEB_MAR_TABLE}

        WHERE month IN ('2026-02', '2026-03')

        GROUP BY
            client_hash_id,
            content_hash_id,
            month
    ),

    february AS (
        SELECT
            client_hash_id,
            content_hash_id,

            gsc_observed_days AS feb_gsc_days,
            ga4_observed_days AS feb_ga4_days,

            avg_daily_impressions AS feb_avg_daily_impressions,
            avg_daily_clicks AS feb_avg_daily_clicks,
            weighted_avg_position AS feb_avg_position,
            avg_daily_sessions AS feb_avg_daily_sessions,
            engagement_rate AS feb_engagement_rate

        FROM monthly

        WHERE month = '2026-02'
    ),

    march AS (
        SELECT
            client_hash_id,
            content_hash_id,

            gsc_observed_days AS march_gsc_days,
            avg_daily_clicks AS march_avg_daily_clicks

        FROM monthly

        WHERE month = '2026-03'
    )

    SELECT
        feb.client_hash_id,
        feb.content_hash_id,
        '2026-02' AS decision_month,

        feb.feb_avg_daily_impressions,
        feb.feb_avg_daily_clicks,
        feb.feb_avg_position,
        feb.feb_avg_daily_sessions,
        feb.feb_engagement_rate,

        mar.march_avg_daily_clicks,

        CASE
            WHEN mar.march_avg_daily_clicks
                 < feb.feb_avg_daily_clicks
            THEN 1
            ELSE 0
        END AS declined_next_month

    FROM february AS feb

    INNER JOIN march AS mar
        ON feb.client_hash_id = mar.client_hash_id
        AND feb.content_hash_id = mar.content_hash_id

    WHERE feb.feb_gsc_days > 0
      AND feb.feb_ga4_days > 0
      AND mar.march_gsc_days > 0

      AND feb.feb_avg_daily_impressions IS NOT NULL
      AND feb.feb_avg_daily_clicks IS NOT NULL
      AND feb.feb_avg_position IS NOT NULL
      AND feb.feb_avg_daily_sessions IS NOT NULL
      AND feb.feb_engagement_rate IS NOT NULL
      AND mar.march_avg_daily_clicks IS NOT NULL
    """
).df()

print("Modelling rows:", len(model_frame))

print("\nLabel counts:")
print(model_frame["declined_next_month"].value_counts())

print("\nLabel percentages:")
print(
    model_frame["declined_next_month"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(model_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modelling rows: 23742

Label counts:
declined_next_month
0    14391
1     9351
Name: count, dtype: int64

Label percentages:
declined_next_month
0    60.61
1    39.39
Name: proportion, dtype: float64


,client_hash_id,content_hash_id,decision_month,feb_avg_daily_impressions,feb_avg_daily_clicks,feb_avg_position,feb_avg_daily_sessions,feb_engagement_rate,march_avg_daily_clicks,declined_next_month
0,client_e547b89c05043229,content_7995404695ee1ffd,2026-02,36.142857,0.107143,28.886364,1.25,0.0,0.034483,1
1,client_e547b89c05043229,content_acf700633f016e5a,2026-02,8.750000,0.000000,8.032653,1.00,0.0,0.000000,0
2,client_e547b89c05043229,content_a2bd730a7cf68316,2026-02,19.678571,0.035714,6.154265,1.00,0.0,0.034483,1
3,client_e547b89c05043229,content_feb20a281a923fc6,2026-02,157.107143,0.035714,0.949079,1.00,0.0,0.000000,1
4,client_e547b89c05043229,content_1f65dce010ac9da9,2026-02,36.571429,0.000000,10.566406,1.00,0.0,0.000000,0


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

This baseline prioritizes pages whose February click-through rate is lower
than what is normally observed for pages at a similar search position.

Pages with high impressions receive higher priority because improving
their search snippet may produce a larger practical impact.

The rule uses only information available in February. March performance
and the `declined_next_month` label are used only to evaluate the completed
ranking.

### Reason codes

- `HIGH_VISIBILITY_LOW_CTR`: The page has high impressions and performs
  below the expected CTR for its search position.
- `LOW_CTR_FOR_POSITION`: The page performs below the expected CTR for
  its search position.
- `HIGH_VISIBILITY`: The page has high impressions but no strong CTR issue.
- `MONITOR`: No strong action signal was observed.

In [3]:
# Create a clean working copy.
decision_df = model_frame.copy()

# Keep rows with usable February search information.
decision_df = decision_df[
    (decision_df["feb_avg_daily_impressions"] > 0)
    & (decision_df["feb_avg_daily_clicks"] >= 0)
    & (decision_df["feb_avg_position"] > 0)
].copy()

# Calculate February click-through rate.
decision_df["feb_ctr"] = (
    decision_df["feb_avg_daily_clicks"]
    / decision_df["feb_avg_daily_impressions"]
)

# Create understandable search-position groups.
position_bins = [0, 3, 10, 20, 50, float("inf")]

position_labels = [
    "positions_1_to_3",
    "positions_4_to_10",
    "positions_11_to_20",
    "positions_21_to_50",
    "positions_51_plus",
]

decision_df["position_bucket"] = pd.cut(
    decision_df["feb_avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True,
)

# Expected CTR is the average CTR of pages in the same position group.
decision_df["expected_ctr"] = (
    decision_df
    .groupby("position_bucket", observed=True)["feb_ctr"]
    .transform("mean")
)

# Negative values mean CTR is below the position-group expectation.
decision_df["ctr_gap"] = (
    decision_df["feb_ctr"]
    - decision_df["expected_ctr"]
)

# Divide pages into three equally sized CTR-gap groups.
decision_df["ctr_signal_bucket"] = pd.qcut(
    decision_df["ctr_gap"].rank(method="first"),
    q=3,
    labels=[
        "lowest_ctr_vs_position",
        "middle_ctr_vs_position",
        "highest_ctr_vs_position",
    ],
)

signal_1_table = (
    decision_df
    .groupby("ctr_signal_bucket", observed=True)
    .agg(
        n=("declined_next_month", "size"),
        average_position=("feb_avg_position", "mean"),
        average_ctr=("feb_ctr", "mean"),
        average_expected_ctr=("expected_ctr", "mean"),
        average_ctr_gap=("ctr_gap", "mean"),
        decline_rate=("declined_next_month", "mean"),
    )
    .reset_index()
)

# Convert CTR and decline values into percentages.
for column in [
    "average_ctr",
    "average_expected_ctr",
    "average_ctr_gap",
    "decline_rate",
]:
    signal_1_table[column] = signal_1_table[column] * 100

print("Signal 1: CTR compared with similar search positions\n")
print(signal_1_table.round(3).to_string(index=False))

Signal 1: CTR compared with similar search positions

      ctr_signal_bucket    n  average_position  average_ctr  average_expected_ctr  average_ctr_gap  decline_rate
 lowest_ctr_vs_position 7901            10.699        0.045                 0.596           -0.551        16.492
 middle_ctr_vs_position 7901            19.359        0.121                 0.453           -0.332        36.008
highest_ctr_vs_position 7901            13.612        1.382                 0.499            0.883        65.840


### Signal 1 verdict: OPPOSITE

Pages with the lowest CTR compared with similar search positions had an
observed next-month decline rate of 16.49%.

Pages with the highest CTR compared with similar search positions had an
observed next-month decline rate of 65.84%.

This is the opposite of my original assumption. Low CTR relative to
position was not associated with a higher decline rate in this data
slice. Therefore, I will not use low CTR as a positive risk signal in the
final baseline rule.

This result is directional and may partly reflect regression to the mean,
page maturity, search intent or other unmeasured factors. It should not
be interpreted as proof that high CTR causes future decline.

In [4]:
# Inspect the engagement-rate distribution first.
print("Engagement-rate summary:")
print(
    decision_df["feb_engagement_rate"]
    .describe(percentiles=[0.25, 0.50, 0.75])
    .round(4)
)

print(
    "\nRows with zero engagement:",
    (decision_df["feb_engagement_rate"] <= 0).sum()
)

# Calculate thresholds using positive engagement values only.
positive_engagement = decision_df.loc[
    decision_df["feb_engagement_rate"] > 0,
    "feb_engagement_rate",
]

low_engagement_threshold = positive_engagement.quantile(0.33)
high_engagement_threshold = positive_engagement.quantile(0.67)

print(
    "\nLow positive-engagement threshold:",
    round(low_engagement_threshold, 4),
)

print(
    "High positive-engagement threshold:",
    round(high_engagement_threshold, 4),
)

# Create understandable engagement groups.
decision_df["engagement_signal_bucket"] = np.select(
    [
        decision_df["feb_engagement_rate"] <= 0,

        decision_df["feb_engagement_rate"].between(
            0,
            low_engagement_threshold,
            inclusive="right",
        ),

        decision_df["feb_engagement_rate"].between(
            low_engagement_threshold,
            high_engagement_threshold,
            inclusive="right",
        ),
    ],
    [
        "zero_engagement",
        "low_positive_engagement",
        "medium_positive_engagement",
    ],
    default="high_positive_engagement",
)

signal_2_table = (
    decision_df
    .groupby("engagement_signal_bucket")
    .agg(
        n=("declined_next_month", "size"),
        average_engagement_rate=(
            "feb_engagement_rate",
            "mean",
        ),
        average_daily_sessions=(
            "feb_avg_daily_sessions",
            "mean",
        ),
        decline_rate=(
            "declined_next_month",
            "mean",
        ),
    )
    .reset_index()
)

signal_2_table["average_engagement_rate"] *= 100
signal_2_table["decline_rate"] *= 100

bucket_order = [
    "zero_engagement",
    "low_positive_engagement",
    "medium_positive_engagement",
    "high_positive_engagement",
]

signal_2_table["engagement_signal_bucket"] = pd.Categorical(
    signal_2_table["engagement_signal_bucket"],
    categories=bucket_order,
    ordered=True,
)

signal_2_table = signal_2_table.sort_values(
    "engagement_signal_bucket"
)

print("\nSignal 2: February engagement rate\n")

print(
    signal_2_table
    .round(3)
    .to_string(index=False)
)

Engagement-rate summary:
count    23703.0000
mean         0.0448
std          0.1288
min          0.0000
25%          0.0000
50%          0.0000
75%          0.0278
max          1.0000
Name: feb_engagement_rate, dtype: float64

Rows with zero engagement: 17364

Low positive-engagement threshold: 0.0667
High positive-engagement threshold: 0.1429

Signal 2: February engagement rate

  engagement_signal_bucket     n  average_engagement_rate  average_daily_sessions  decline_rate
           zero_engagement 17364                    0.000                   1.672        35.654
   low_positive_engagement  2104                    4.313                   3.441        54.800
medium_positive_engagement  2199                   10.202                   2.359        51.296
  high_positive_engagement  2036                   36.653                   1.309        43.124


### Signal 2 verdict: MIXED

Pages with low positive engagement had an observed next-month decline rate
of 54.80%, while pages with medium positive engagement had a decline rate
of 51.30%.

Pages with high positive engagement had a lower decline rate of 43.12%.
However, pages with zero engagement had the lowest decline rate at 35.65%.

The relationship is therefore not consistent across all engagement
groups. I classify this signal as MIXED.

Low-to-medium positive engagement may provide a weak supporting signal,
but engagement should not be used alone to prioritize pages. Zero
engagement is not automatically treated as a decline-risk signal.

### Final baseline rule

This baseline prioritizes pages whose February CTR is unusually strong
compared with pages at a similar search position.

In this data slice, pages with stronger CTR relative to position had a
higher observed next-month decline rate. These pages should be monitored
to protect existing search traffic rather than automatically changed.

Low-to-medium positive engagement is used only as a weak supporting
signal because its relationship with decline was mixed.

High impressions increase priority because a decline on a highly visible
page may have a larger practical impact.

The rule uses only February information to calculate the score. March
performance and `declined_next_month` are used only after ranking for
evaluation.

### Reason codes

- `HIGH_VISIBILITY_ABOVE_EXPECTED_CTR`: The page has above-expected CTR
  and high search visibility.
- `ABOVE_EXPECTED_CTR_AND_ENGAGEMENT_WATCH`: The page has above-expected
  CTR and low-to-medium positive engagement.
- `ABOVE_EXPECTED_CTR`: The page's CTR is above the expectation for its
  search-position group.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# ---------------------------------------------------------
# Thresholds calculated using February information only
# ---------------------------------------------------------

# Top third of CTR performance relative to position.
strong_ctr_threshold = 0.0
# Top quarter by February impressions.
visibility_threshold = (
    decision_df["feb_avg_daily_impressions"].quantile(0.75)
)

# Signal 2 showed that low and medium positive engagement
# had higher observed decline rates than high engagement.
engagement_watch_threshold = high_engagement_threshold


# ---------------------------------------------------------
# Create transparent rule flags
# ---------------------------------------------------------

decision_df["strong_ctr_vs_position"] = (
    decision_df["ctr_gap"] >= strong_ctr_threshold
).astype(int)

decision_df["low_positive_engagement"] = (
    (decision_df["feb_engagement_rate"] > 0)
    & (
        decision_df["feb_engagement_rate"]
        <= engagement_watch_threshold
    )
).astype(int)

decision_df["high_visibility"] = (
    decision_df["feb_avg_daily_impressions"]
    >= visibility_threshold
).astype(int)


# ---------------------------------------------------------
# Create a simple, manually weighted baseline score
# ---------------------------------------------------------

decision_df["action_score"] = (
    3 * decision_df["strong_ctr_vs_position"]
    + 1 * decision_df["low_positive_engagement"]
    + 2 * decision_df["high_visibility"]
)


# ---------------------------------------------------------
# Attach one reason code to every page
# ---------------------------------------------------------

reason_conditions = [
    (
        (decision_df["strong_ctr_vs_position"] == 1)
        & (decision_df["high_visibility"] == 1)
    ),
    (
        (decision_df["strong_ctr_vs_position"] == 1)
        & (decision_df["low_positive_engagement"] == 1)
    ),
    decision_df["strong_ctr_vs_position"] == 1,
    decision_df["low_positive_engagement"] == 1,
    decision_df["high_visibility"] == 1,
]

reason_choices = [
    "HIGH_VISIBILITY_ABOVE_EXPECTED_CTR",
    "ABOVE_EXPECTED_CTR_AND_ENGAGEMENT_WATCH",
    "ABOVE_EXPECTED_CTR",
    "LOW_POSITIVE_ENGAGEMENT",
    "HIGH_VISIBILITY_ONLY",
]

decision_df["reason_code"] = np.select(
    reason_conditions,
    reason_choices,
    default="MONITOR",
)


# ---------------------------------------------------------
# Add a recommended action
# ---------------------------------------------------------

action_conditions = [
    (
        (decision_df["strong_ctr_vs_position"] == 1)
        & (decision_df["high_visibility"] == 1)
    ),
    decision_df["strong_ctr_vs_position"] == 1,
    decision_df["low_positive_engagement"] == 1,
    decision_df["high_visibility"] == 1,
]

action_choices = [
    "PROTECT_AND_MONITOR",
    "MONITOR_FOR_DECLINE",
    "REVIEW_ENGAGEMENT",
    "MONITOR_HIGH_VISIBILITY",
]

decision_df["recommended_action"] = np.select(
    action_conditions,
    action_choices,
    default="MONITOR",
)


# ---------------------------------------------------------
# Rank the complete queue
# ---------------------------------------------------------

ranked_queue = (
    decision_df
    .sort_values(
        by=[
            "action_score",
            "feb_avg_daily_impressions",
            "ctr_gap",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

ranked_queue.insert(
    0,
    "priority_rank",
    np.arange(1, len(ranked_queue) + 1),
)


# ---------------------------------------------------------
# Display thresholds and top results
# ---------------------------------------------------------

print(
    "Strong CTR-gap threshold:",
    round(strong_ctr_threshold * 100, 3),
    "percentage points",
)

print(
    "High-visibility threshold:",
    round(visibility_threshold, 3),
    "average daily impressions",
)

print(
    "Engagement-watch threshold:",
    round(engagement_watch_threshold * 100, 3),
    "%",
)

print("\nAction-score distribution:")
print(
    ranked_queue["action_score"]
    .value_counts()
    .sort_index(ascending=False)
)

top_columns = [
    "priority_rank",
    "content_hash_id",
    "action_score",
    "reason_code",
    "recommended_action",
    "feb_avg_daily_impressions",
    "feb_avg_position",
    "feb_ctr",
    "expected_ctr",
    "ctr_gap",
    "feb_engagement_rate",
]

display(ranked_queue[top_columns].head(10))

Strong CTR-gap threshold: 0.0 percentage points
High-visibility threshold: 88.482 average daily impressions
Engagement-watch threshold: 14.286 %

Action-score distribution:
action_score
6     1050
5      228
4      641
3     5119
2     2651
1      615
0    13399
Name: count, dtype: int64


,priority_rank,content_hash_id,action_score,reason_code,recommended_action,feb_avg_daily_impressions,feb_avg_position,feb_ctr,expected_ctr,ctr_gap,feb_engagement_rate
0,1,content_c9a0c2fdbdbfb562,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,5079.107143,1.906880,0.011286,0.008063,0.003222,0.035443
1,2,content_eadb33b5df496f4a,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,3387.571429,2.554295,0.011839,0.008063,0.003776,0.062392
2,3,content_f7c9fcc26f6e23c1,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,2770.892857,6.133660,0.012013,0.006205,0.005808,0.061004
3,4,content_40baa8f1016f5742,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,2281.857143,5.270973,0.008984,0.006205,0.002779,0.050063
4,5,content_7de2236ec61ba4c5,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,2092.535714,4.916404,0.010070,0.006205,0.003865,0.048059
5,6,content_3508adb0f05ec0b9,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1588.821429,3.226516,0.006541,0.006205,0.000336,0.075085
6,7,content_86c036a549cbbfd5,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1441.928571,2.607123,0.010205,0.008063,0.002141,0.030242
7,8,content_6f50bf2780b5d040,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1436.321429,18.102966,0.007857,0.004057,0.003800,0.072340
8,9,content_425a23ff83f9f448,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1431.642857,5.350372,0.011276,0.006205,0.005071,0.077320
9,10,content_5460a0e854602a47,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1419.035714,24.532706,0.004606,0.003870,0.000736,0.077266


In [7]:
# Check whether the corrected CTR flag still separates
# pages with different observed outcomes.

ctr_rule_check = (
    ranked_queue
    .groupby("strong_ctr_vs_position")
    .agg(
        n=("declined_next_month", "size"),
        average_ctr_gap=("ctr_gap", "mean"),
        decline_rate=("declined_next_month", "mean"),
    )
    .reset_index()
)

ctr_rule_check["average_ctr_gap"] *= 100
ctr_rule_check["decline_rate"] *= 100

ctr_rule_check["group"] = ctr_rule_check[
    "strong_ctr_vs_position"
].map(
    {
        0: "CTR not above expectation",
        1: "CTR above expectation",
    }
)

print("Corrected CTR rule check:\n")

print(
    ctr_rule_check[
        [
            "group",
            "n",
            "average_ctr_gap",
            "decline_rate",
        ]
    ]
    .round(3)
    .to_string(index=False)
)

Corrected CTR rule check:

                    group     n  average_ctr_gap  decline_rate
CTR not above expectation 18662           -0.390        31.219
    CTR above expectation  5041            1.445        69.907


### Corrected CTR rule verdict: CONFIRMED

Pages whose February CTR was above the expected CTR for their
search-position group had an observed next-month decline rate of 69.91%.

Pages whose CTR was not above expectation had an observed decline rate
of 31.22%.

This confirms that above-expected CTR is a useful directional signal for
this baseline. It may help identify currently successful pages whose
existing search traffic should be protected and monitored.

This result does not establish causation. Differences in search intent,
seasonality, page maturity, query mix or regression to the mean may also
influence the observed outcome.

In [8]:
from pathlib import Path

# Columns safe to include in the operational action queue.
export_columns = [
    "priority_rank",
    "client_hash_id",
    "content_hash_id",
    "decision_month",
    "action_score",
    "reason_code",
    "recommended_action",
    "feb_avg_daily_impressions",
    "feb_avg_daily_clicks",
    "feb_avg_position",
    "feb_avg_daily_sessions",
    "feb_engagement_rate",
    "feb_ctr",
    "expected_ctr",
    "ctr_gap",
    "strong_ctr_vs_position",
    "low_positive_engagement",
    "high_visibility",
]

baseline_action_queue = ranked_queue[export_columns].copy()

output_path = Path(
    "work/outputs/baseline_action_score.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

baseline_action_queue.to_csv(
    output_path,
    index=False,
)

print("Ranked rows saved:", len(baseline_action_queue))
print("Output path:", output_path)

print("\nColumns written to the CSV:")
print(baseline_action_queue.columns.tolist())

display(baseline_action_queue.head(10))

Ranked rows saved: 23703
Output path: work/outputs/baseline_action_score.csv

Columns written to the CSV:
['priority_rank', 'client_hash_id', 'content_hash_id', 'decision_month', 'action_score', 'reason_code', 'recommended_action', 'feb_avg_daily_impressions', 'feb_avg_daily_clicks', 'feb_avg_position', 'feb_avg_daily_sessions', 'feb_engagement_rate', 'feb_ctr', 'expected_ctr', 'ctr_gap', 'strong_ctr_vs_position', 'low_positive_engagement', 'high_visibility']


,priority_rank,client_hash_id,content_hash_id,decision_month,action_score,reason_code,recommended_action,feb_avg_daily_impressions,feb_avg_daily_clicks,feb_avg_position,feb_avg_daily_sessions,feb_engagement_rate,feb_ctr,expected_ctr,ctr_gap,strong_ctr_vs_position,low_positive_engagement,high_visibility
0,1,client_e547b89c05043229,content_c9a0c2fdbdbfb562,2026-02,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,5079.107143,57.321429,1.906880,14.107143,0.035443,0.011286,0.008063,0.003222,1,1,1
1,2,client_e547b89c05043229,content_eadb33b5df496f4a,2026-02,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,3387.571429,40.107143,2.554295,20.607143,0.062392,0.011839,0.008063,0.003776,1,1,1
2,3,client_23a62021009f63c4,content_f7c9fcc26f6e23c1,2026-02,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,2770.892857,33.285714,6.133660,46.250000,0.061004,0.012013,0.006205,0.005808,1,1,1
3,4,client_23a62021009f63c4,content_40baa8f1016f5742,2026-02,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,2281.857143,20.500000,5.270973,28.535714,0.050063,0.008984,0.006205,0.002779,1,1,1
4,5,client_e547b89c05043229,content_7de2236ec61ba4c5,2026-02,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,2092.535714,21.071429,4.916404,19.321429,0.048059,0.010070,0.006205,0.003865,1,1,1
5,6,client_e547b89c05043229,content_3508adb0f05ec0b9,2026-02,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1588.821429,10.392857,3.226516,10.464286,0.075085,0.006541,0.006205,0.000336,1,1,1
6,7,client_e547b89c05043229,content_86c036a549cbbfd5,2026-02,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1441.928571,14.714286,2.607123,17.714286,0.030242,0.010205,0.008063,0.002141,1,1,1
7,8,client_23a62021009f63c4,content_6f50bf2780b5d040,2026-02,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1436.321429,11.285714,18.102966,16.785714,0.072340,0.007857,0.004057,0.003800,1,1,1
8,9,client_e547b89c05043229,content_425a23ff83f9f448,2026-02,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1431.642857,16.142857,5.350372,13.857143,0.077320,0.011276,0.006205,0.005071,1,1,1
9,10,client_23a62021009f63c4,content_5460a0e854602a47,2026-02,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1419.035714,6.535714,24.532706,144.214286,0.077266,0.004606,0.003870,0.000736,1,1,1


In [9]:
# Evaluate the completed ranking.
# Future outcomes are allowed here because scoring is already finished.

K = 20

top_k = ranked_queue.head(K).copy()

base_rate = ranked_queue[
    "declined_next_month"
].mean()

precision_at_k = top_k[
    "declined_next_month"
].mean()

lift_over_base_rate = (
    precision_at_k / base_rate
    if base_rate > 0
    else np.nan
)

correct_picks = int(
    top_k["declined_next_month"].sum()
)

print(f"Evaluation size K: {K}")
print(f"Correct decline picks: {correct_picks} of {K}")
print(f"Overall decline base rate: {base_rate:.2%}")
print(f"Precision@{K}: {precision_at_k:.2%}")
print(f"Lift over base rate: {lift_over_base_rate:.2f}x")

Evaluation size K: 20
Correct decline picks: 13 of 20
Overall decline base rate: 39.45%
Precision@20: 65.00%
Lift over base rate: 1.65x


### Baseline evaluation

The overall next-month decline rate was 39.45%.

The baseline achieved Precision@20 of 65.00%, correctly identifying 13
declining pages among its top 20 recommendations.

This represents a lift of 1.65 times over the overall base rate. The
result suggests that the transparent baseline provides a useful starting
ranking that a future machine-learning model should attempt to improve.

However, seven of the top 20 pages did not decline. Therefore, this
baseline should be treated as decision support rather than an automatic
content-action system.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
# Review the 20 highest-ranked recommendations.

top_20_review = ranked_queue.head(20).copy()


def explain_selection(row):
    explanations = []

    if row["strong_ctr_vs_position"] == 1:
        explanations.append(
            "February CTR was above the expectation for its "
            "search-position group"
        )

    if row["high_visibility"] == 1:
        explanations.append(
            "the page had high average daily impressions"
        )

    if row["low_positive_engagement"] == 1:
        explanations.append(
            "engagement was positive but within the watch range"
        )

    if not explanations:
        return "No strong priority flag was present."

    return "; ".join(explanations) + "."


def assign_confidence(row):
    if (
        row["strong_ctr_vs_position"] == 1
        and row["high_visibility"] == 1
        and row["ctr_gap"] > 0
    ):
        return (
            "High: above-expected CTR and high visibility "
            "support the recommendation."
        )

    if row["strong_ctr_vs_position"] == 1:
        return (
            "Medium: CTR is above expectation, but fewer "
            "supporting signals are present."
        )

    return (
        "Low: the recommendation depends mainly on a weaker "
        "supporting signal."
    )


def explain_possible_failure(row):
    possible_failures = []

    if row["strong_ctr_vs_position"] == 1:
        possible_failures.append(
            "the high CTR may come from branded queries, seasonality "
            "or a temporary query mix"
        )

    if row["high_visibility"] == 1:
        possible_failures.append(
            "high impressions alone do not guarantee that performance "
            "will decline"
        )

    if row["low_positive_engagement"] == 1:
        possible_failures.append(
            "engagement may be affected by tracking quality, low "
            "session volume or page purpose"
        )

    possible_failures.append(
        "monthly averages may hide daily volatility or recent changes"
    )

    return "; ".join(possible_failures) + "."


top_20_review["why_selected"] = top_20_review.apply(
    explain_selection,
    axis=1,
)

top_20_review["confidence_note"] = top_20_review.apply(
    assign_confidence,
    axis=1,
)

top_20_review["what_would_make_it_wrong"] = top_20_review.apply(
    explain_possible_failure,
    axis=1,
)

# Show the later outcome only for evaluation.
top_20_review["observed_outcome"] = np.where(
    top_20_review["declined_next_month"] == 1,
    "DECLINED",
    "DID_NOT_DECLINE",
)

review_columns = [
    "priority_rank",
    "content_hash_id",
    "recommended_action",
    "reason_code",
    "action_score",
    "why_selected",
    "confidence_note",
    "what_would_make_it_wrong",
    "observed_outcome",
]

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 30)

display(top_20_review[review_columns])

,priority_rank,content_hash_id,recommended_action,reason_code,action_score,why_selected,confidence_note,what_would_make_it_wrong,observed_outcome
0,1,content_c9a0c2fdbdbfb562,PROTECT_AND_MONITOR,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,6,February CTR was above the expectation for its search-position group; the page had high average daily impressions; engagement was positive but within the watch range.,High: above-expected CTR and high visibility support the recommendation.,"the high CTR may come from branded queries, seasonality or a temporary query mix; high impressions alone do not guarantee that performance will decline; engagement may be affected by tracking qual...",DECLINED
1,2,content_eadb33b5df496f4a,PROTECT_AND_MONITOR,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,6,February CTR was above the expectation for its search-position group; the page had high average daily impressions; engagement was positive but within the watch range.,High: above-expected CTR and high visibility support the recommendation.,"the high CTR may come from branded queries, seasonality or a temporary query mix; high impressions alone do not guarantee that performance will decline; engagement may be affected by tracking qual...",DID_NOT_DECLINE
2,3,content_f7c9fcc26f6e23c1,PROTECT_AND_MONITOR,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,6,February CTR was above the expectation for its search-position group; the page had high average daily impressions; engagement was positive but within the watch range.,High: above-expected CTR and high visibility support the recommendation.,"the high CTR may come from branded queries, seasonality or a temporary query mix; high impressions alone do not guarantee that performance will decline; engagement may be affected by tracking qual...",DECLINED
3,4,content_40baa8f1016f5742,PROTECT_AND_MONITOR,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,6,February CTR was above the expectation for its search-position group; the page had high average daily impressions; engagement was positive but within the watch range.,High: above-expected CTR and high visibility support the recommendation.,"the high CTR may come from branded queries, seasonality or a temporary query mix; high impressions alone do not guarantee that performance will decline; engagement may be affected by tracking qual...",DECLINED
4,5,content_7de2236ec61ba4c5,PROTECT_AND_MONITOR,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,6,February CTR was above the expectation for its search-position group; the page had high average daily impressions; engagement was positive but within the watch range.,High: above-expected CTR and high visibility support the recommendation.,"the high CTR may come from branded queries, seasonality or a temporary query mix; high impressions alone do not guarantee that performance will decline; engagement may be affected by tracking qual...",DID_NOT_DECLINE
5,6,content_3508adb0f05ec0b9,PROTECT_AND_MONITOR,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,6,February CTR was above the expectation for its search-position group; the page had high average daily impressions; engagement was positive but within the watch range.,High: above-expected CTR and high visibility support the recommendation.,"the high CTR may come from branded queries, seasonality or a temporary query mix; high impressions alone do not guarantee that performance will decline; engagement may be affected by tracking qual...",DECLINED
6,7,content_86c036a549cbbfd5,PROTECT_AND_MONITOR,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,6,February CTR was above the expectation for its search-position group; the page had high average daily impressions; engagement was positive but within the watch range.,High: above-expected CTR and high visibility support the recommendation.,"the high CTR may come from branded queries, seasonality or a temporary query mix; high impressions alone do not guarantee that performance will decline; engagement may be affected by tracking qual...",DECLINED
7,8,content_6f50bf2780b5d040,PROTECT_AND_MONITOR,HIGH_VISIBILITY_ABOVE_EXPECTED_C

In [11]:
for _, row in top_20_review.iterrows():
    print("=" * 90)
    print(f"Priority rank: {row['priority_rank']}")
    print(f"Content ID: {row['content_hash_id']}")
    print(f"Action: {row['recommended_action']}")
    print(f"Reason code: {row['reason_code']}")
    print(f"Score: {row['action_score']}")
    print(f"Why selected: {row['why_selected']}")
    print(f"Confidence: {row['confidence_note']}")
    print(
        "What could make it wrong:",
        row["what_would_make_it_wrong"],
    )
    print(f"Observed outcome: {row['observed_outcome']}")

Priority rank: 1
Content ID: content_c9a0c2fdbdbfb562
Action: PROTECT_AND_MONITOR
Reason code: HIGH_VISIBILITY_ABOVE_EXPECTED_CTR
Score: 6
Why selected: February CTR was above the expectation for its search-position group; the page had high average daily impressions; engagement was positive but within the watch range.
Confidence: High: above-expected CTR and high visibility support the recommendation.
What could make it wrong: the high CTR may come from branded queries, seasonality or a temporary query mix; high impressions alone do not guarantee that performance will decline; engagement may be affected by tracking quality, low session volume or page purpose; monthly averages may hide daily volatility or recent changes.
Observed outcome: DECLINED
Priority rank: 2
Content ID: content_eadb33b5df496f4a
Action: PROTECT_AND_MONITOR
Reason code: HIGH_VISIBILITY_ABOVE_EXPECTED_CTR
Score: 6
Why selected: February CTR was above the expectation for its search-position group; the page had high av

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
# Weak picks are top-ranked pages that did not decline
# in the following month.

weak_picks = top_20_review[
    top_20_review["declined_next_month"] == 0
].copy()

weak_picks["weak_pick_explanation"] = (
    "The baseline ranked this page highly because it combined "
    "above-expected CTR, high visibility and watch-range engagement. "
    "However, the page did not decline in March. This suggests that "
    "the rule cannot fully account for factors such as seasonality, "
    "branded traffic, query intent, page maturity or daily volatility."
)

weak_pick_columns = [
    "priority_rank",
    "content_hash_id",
    "action_score",
    "reason_code",
    "recommended_action",
    "feb_avg_daily_impressions",
    "feb_ctr",
    "expected_ctr",
    "ctr_gap",
    "feb_engagement_rate",
    "observed_outcome",
    "weak_pick_explanation",
]

print("Weak picks in the top 20:", len(weak_picks))
print(
    "Weak-pick rate:",
    f"{len(weak_picks) / len(top_20_review):.2%}",
)

display(weak_picks[weak_pick_columns])

Weak picks in the top 20: 7
Weak-pick rate: 35.00%


,priority_rank,content_hash_id,action_score,reason_code,recommended_action,feb_avg_daily_impressions,feb_ctr,expected_ctr,ctr_gap,feb_engagement_rate,observed_outcome,weak_pick_explanation
1,2,content_eadb33b5df496f4a,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,3387.571429,0.011839,0.008063,0.003776,0.062392,DID_NOT_DECLINE,"The baseline ranked this page highly because it combined above-expected CTR, high visibility and watch-range engagement. However, the page did not decline in March. This suggests that the rule can..."
4,5,content_7de2236ec61ba4c5,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,2092.535714,0.010070,0.006205,0.003865,0.048059,DID_NOT_DECLINE,"The baseline ranked this page highly because it combined above-expected CTR, high visibility and watch-range engagement. However, the page did not decline in March. This suggests that the rule can..."
10,11,content_18f0847d6628f8c6,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1379.071429,0.008676,0.006205,0.002471,0.129683,DID_NOT_DECLINE,"The baseline ranked this page highly because it combined above-expected CTR, high visibility and watch-range engagement. However, the page did not decline in March. This suggests that the rule can..."
12,13,content_2d7b116d68e8b40b,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1302.107143,0.007022,0.006205,0.000817,0.046832,DID_NOT_DECLINE,"The baseline ranked this page highly because it combined above-expected CTR, high visibility and watch-range engagement. However, the page did not decline in March. This suggests that the rule can..."
14,15,content_972ccc7535cb4bd5,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1268.571429,0.008840,0.008063,0.000777,0.110119,DID_NOT_DECLINE,"The baseline ranked this page highly because it combined above-expected CTR, high visibility and watch-range engagement. However, the page did not decline in March. This suggests that the rule can..."
16,17,content_df960beba08b53e3,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1156.928571,0.007717,0.006205,0.001513,0.058462,DID_NOT_DECLINE,"The baseline ranked this page highly because it combined above-expected CTR, high visibility and watch-range engagement. However, the page did not decline in March. This suggests that the rule can..."
18,19,content_3b33966873e77d42,6,HIGH_VISIBILITY_ABOVE_EXPECTED_CTR,PROTECT_AND_MONITOR,1144.214286,0.005650,0.004057,0.001592,0.053942,DID_NOT_DECLINE,"The baseline ranked this page highly because it combined above-expected CTR, high visibility and watch-range engagement. However, the page did not decline in March. This suggests that the rule can..."


### Weak-pick review

Seven of the top 20 recommendations did not decline in the following
month, giving the baseline a weak-pick rate of 35%.

These pages received the maximum score because they combined
above-expected CTR, high visibility and positive engagement within the
watch range. However, those signals did not guarantee a future decline.

One weakness is that many pages received the same score and reason code.
Their final order was therefore influenced mainly by average daily
impressions. High impressions increase potential business impact, but
they do not necessarily increase the probability of decline.

Other possible explanations include branded search traffic, seasonality,
query-intent differences, page maturity, analytics quality and daily
variation hidden by monthly averages.

The baseline is therefore suitable for human review and decision support,
not automatic content changes.

In [13]:
# ---------------------------------------------------------
# Leakage check
# ---------------------------------------------------------

# These are the only raw or derived inputs used to calculate
# the baseline score.
score_input_columns = [
    "feb_avg_daily_impressions",
    "feb_avg_daily_clicks",
    "feb_avg_position",
    "feb_avg_daily_sessions",
    "feb_engagement_rate",
    "feb_ctr",
    "expected_ctr",
    "ctr_gap",
    "strong_ctr_vs_position",
    "low_positive_engagement",
    "high_visibility",
]

# Terms that indicate future information, target leakage
# or product-generated recommendations.
banned_terms = [
    "march",
    "future",
    "declined",
    "next_month",
    "label",
    "target",
    "recommendation_flag",
    "product_flag",
    "trend_direction",
    "trend_pct",
]

score_leakage_columns = [
    column
    for column in score_input_columns
    if any(
        term in column.lower()
        for term in banned_terms
    )
]

# Check the operational CSV separately.
export_leakage_columns = [
    column
    for column in baseline_action_queue.columns
    if any(
        term in column.lower()
        for term in [
            "march",
            "declined",
            "next_month",
            "label",
            "target",
        ]
    )
]

print("Score inputs checked:", score_input_columns)
print("\nPotential leakage in score:", score_leakage_columns)
print(
    "Potential future fields in exported queue:",
    export_leakage_columns,
)

assert not score_leakage_columns, (
    f"Leakage found in score inputs: {score_leakage_columns}"
)

assert not export_leakage_columns, (
    f"Future fields found in CSV export: {export_leakage_columns}"
)

assert "march_avg_daily_clicks" not in score_input_columns
assert "declined_next_month" not in score_input_columns

print("\nLeakage check passed.")
print("The action score uses February information only.")
print(
    "March clicks and declined_next_month are used only "
    "after ranking for evaluation."
)

Score inputs checked: ['feb_avg_daily_impressions', 'feb_avg_daily_clicks', 'feb_avg_position', 'feb_avg_daily_sessions', 'feb_engagement_rate', 'feb_ctr', 'expected_ctr', 'ctr_gap', 'strong_ctr_vs_position', 'low_positive_engagement', 'high_visibility']

Potential leakage in score: []
Potential future fields in exported queue: []

Leakage check passed.
The action score uses February information only.
March clicks and declined_next_month are used only after ranking for evaluation.


### Leakage conclusion

The action score uses only information available during February.

March average daily clicks and the `declined_next_month` outcome were not
used to calculate page flags, weights, scores, reason codes or ranking
positions. They were used only after ranking to evaluate Precision@20 and
identify weak recommendations.

The exported operational queue also excludes March fields and the future
outcome label.

No FlyRank product recommendation flags, future-window features or
label-derived features were included in the baseline score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.